In [0]:


# Load data from your Silver table
df = spark.table("predictivemaintenancecatalog.gold_data.gold_data_historical_stratified")
display(df)



In [0]:
%pip install seaborn

## Pair plot for everything

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt
# Select numerical features and target
columns = [
    "Air_temperature_K",
    "Process_temperature_K",
    "Rotational_speed_rpm",
    "Torque_Nm",
    "Tool_wear_min",
    "Machine_failure",
]

# Convert to Pandas
pdf = df.toPandas()

# Create pairplot
sns.pairplot(
    pdf[columns],
    hue="Machine_failure",
    diag_kind="hist",
    corner=True,
    plot_kws={"alpha": 0.5}
)

plt.show()


## feature engineering
Mechanical power
Temperature difference
Torque x tool wear
Spead x tool wear
Power x tool wear


In [0]:
import math

#df_features = pdf.copy()

pdf["Mechanical_power_W"] = (
    (2 * math.pi / 60)
    * pdf["Rotational_speed_rpm"]
    * pdf["Torque_Nm"]
)
pdf["Temperature_difference_K"] = (
    pdf["Process_temperature_K"]
    - pdf["Air_temperature_K"]
)
pdf["Torque_wear"] = pdf["Torque_Nm"] * pdf["Tool_wear_min"]
pdf["Speed_wear"] = pdf["Rotational_speed_rpm"] * pdf["Tool_wear_min"]
pdf["Power_wear"] = pdf["Mechanical_power_W"] * pdf["Tool_wear_min"]

display(pdf)

In [0]:
print(pdf.columns.tolist())

## create a balanced pair plot

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

##columns = ["Air_temperature_K","Process_temperature_K","Rotational_speed_rpm","Torque_Nm","Tool_wear_min","Machine_failure","TWF","HDF","PWF","OSF", "RNF"]

# Convert to Pandas
#pdf = df.select(*columns).toPandas()

# Mechanical_power_W is not in the source table, so compute it here (same formula as the feature cell)
#pdf["Mechanical_power_W"] = (
    #2 * np.pi
    #* pdf["Rotational_speed_rpm"]
    #* pdf["Torque_Nm"]
    #/ 60
#)

df_healthy = pdf[pdf["Machine_failure"] == 0]
df_failed = pdf[pdf["Machine_failure"] == 1]

n = min(len(df_healthy), len(df_failed))

pdf_balanced = pd.concat([
    df_healthy.sample(n=n, random_state=42),
    df_failed.sample(n=n, random_state=42)
])

hues= ["Machine_failure","TWF","HDF","PWF","OSF", "RNF"]
plot_features = ["Air_temperature_K","Process_temperature_K","Rotational_speed_rpm","Torque_Nm","Tool_wear_min","Mechanical_power_W",'Temperature_difference_K', 'Torque_wear', 'Speed_wear', 'Power_wear']
for hue in hues:
    g = sns.pairplot(
        pdf_balanced,
        vars=plot_features,
        hue=hue,
        corner=True,
        diag_kind="hist",
        plot_kws={"alpha": 0.5, "s": 25}
    )

    # Add title
    g.fig.suptitle(
        hue,
        fontsize=18,
        fontweight="bold",
        y=1.02
    )

    plt.show()

## Failure-rate plots

In [0]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

features =  ["Air_temperature_K","Process_temperature_K","Rotational_speed_rpm","Torque_Nm","Tool_wear_min","Mechanical_power_W",'Temperature_difference_K', 'Torque_wear', 'Speed_wear', 'Power_wear']

fig, axes = plt.subplots(4, 3, figsize=(16, 15))
axes = axes.flatten()

for i, feature in enumerate(features):

    temp = pdf.copy()  # Original unbalanced data

    temp["bin"] = pd.qcut(
        temp[feature],
        q=10,
        duplicates="drop"
    )

    failure_rate = (
        temp.groupby("bin", observed=True)["Machine_failure"]
        .agg(["mean", "size"])
        .reset_index()
    )

    failure_rate["midpoint"] = failure_rate["bin"].apply(
        lambda x: x.mid
    ).astype(float)

    sns.lineplot(
        data=failure_rate,
        x="midpoint",
        y="mean",
        marker="o",
        ax=axes[i]
    )

    axes[i].set_title(feature)
    axes[i].set_ylabel("Failure rate")
    axes[i].set_xlabel(feature)

axes[-1].axis("off")

plt.tight_layout()
plt.show()

## Correlation heatmap

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

features =  ["Air_temperature_K","Process_temperature_K","Rotational_speed_rpm","Torque_Nm","Tool_wear_min","Mechanical_power_W",'Temperature_difference_K', 'Torque_wear', 'Speed_wear', 'Power_wear']

# Calculate Pearson correlation matrix
corr_matrix = pdf[features].corr(method="pearson")

# Plot heatmap
plt.figure(figsize=(10, 8))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5
)

plt.title(
    "AI4I Predictive Maintenance - Correlation Heatmap",
    fontsize=16,
    fontweight="bold",
    pad=20
)

plt.tight_layout()
plt.show()

## Boxplots by failure status

### Machine failure

In [0]:
display(pdf)

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

features = ["Air_temperature_K","Process_temperature_K","Rotational_speed_rpm","Torque_Nm","Tool_wear_min","Mechanical_power_W",'Temperature_difference_K', 'Torque_wear', 'Speed_wear', 'Power_wear']

fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()

for i, feature in enumerate(features):

    sns.boxplot(
        data=pdf,
        x="Machine_failure",
        y=feature,
        hue="Machine_failure",
        palette="Set2",
        legend=False,
        ax=axes[i],
        showfliers=True
    )

    axes[i].set_title(
        feature.replace("_", " "),
        fontsize=13,
        fontweight="bold"
    )

    axes[i].set_xlabel("Machine Failure (0 = No, 1 = Yes)")
    axes[i].set_ylabel(feature)

fig.suptitle(
    "AI4I Predictive Maintenance - Features by Failure Status",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

### all other failures

In [0]:
failure_modes = ["TWF", "HDF", "PWF", "OSF", "RNF"]

features =  ["Air_temperature_K","Process_temperature_K","Rotational_speed_rpm","Torque_Nm","Tool_wear_min","Mechanical_power_W",'Temperature_difference_K', 'Torque_wear', 'Speed_wear', 'Power_wear']

for failure in failure_modes:

    fig, axes = plt.subplots(2, 5, figsize=(24, 8))
    axes = axes.flatten()

    for i, feature in enumerate(features):

        sns.boxplot(
            data=pdf,
            x=failure,
            y=feature,
            hue=failure,
            palette="Set2",
            legend=False,
            ax=axes[i]
        )

        axes[i].set_title(feature)

    fig.suptitle(
        f"Operating Conditions by {failure} Failure Status",
        fontsize=16,
        fontweight="bold"
    )

    plt.tight_layout()
    plt.show()
    plt.close(fig)

## check how many overlapping failure types we have

In [0]:
from pyspark.sql import functions as F

failure_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]

df_check = pdf.withColumn(
    "failure_mode_count",
    sum(F.col(c) for c in failure_cols)
)

df_check.groupBy("failure_mode_count").count().orderBy(
    "failure_mode_count"
).display()

In [0]:
from pyspark.sql import functions as F

failure_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]

df_check = df_check.withColumn(
    "failure_type",
    F.when(F.col("failure_mode_count") == 0, "Normal")
     .when(F.col("failure_mode_count") > 1, "Multiple")
     .otherwise(
         F.coalesce(
             *[
                 F.when(F.col(c) == 1, F.lit(c))
                 for c in failure_cols
             ]
         )
     )
)

display(
    df_check.groupBy("failure_type")
    .count()
    .orderBy(F.desc("count"))
)

## Write new features to table

In [0]:
display(pdf)

In [0]:
#spark.sql("CREATE SCHEMA IF NOT EXISTS predictivemaintenancecatalog.gold_data")

spark.createDataFrame(pdf).write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("predictivemaintenancecatalog.gold_data.all_features")

display(spark.table("predictivemaintenancecatalog.gold_data.all_features").limit(5))